# Пайплайн POI (задания 1–5)

**Город:** Москва

Нужна **вся папка проекта** (не только ноутбук): `src/features.py`, `data/places.csv`.

В Colab (первая код-ячейка): укажите `REPO_URL` на git.miem.hse.ru или загрузите zip репозитория.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    %pip install -q pandas matplotlib seaborn scikit-learn
except ImportError:
    IN_COLAB = False

import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")

REPO_URL = "https://github.com/AnastasiaBe1ousova/poi_places_pipeline.git"

def find_project_root():
    paths = []
    if IN_COLAB:
        paths.append(Path("/content/poi_places_pipeline"))
        paths.extend(p.parent.parent for p in Path("/content").rglob("features.py") if p.parent.name == "src")
    paths.extend([Path.cwd(), *Path.cwd().parents])
    for p in paths:
        if (p / "data" / "places.csv").is_file() and (p / "src" / "features.py").is_file():
            return p.resolve()
    return None

if IN_COLAB and REPO_URL and not find_project_root():
    !git clone {REPO_URL} /content/poi_places_pipeline

ROOT = find_project_root()
if ROOT is None:
    raise FileNotFoundError(
        "Не найден проект (нужны data/places.csv и src/features.py). "
        "Клонируйте репозиторий целиком или задайте REPO_URL."
    )

feat_file = ROOT / "src" / "features.py"
spec = importlib.util.spec_from_file_location("features", feat_file)
features_mod = importlib.util.module_from_spec(spec)
sys.modules["features"] = features_mod
spec.loader.exec_module(features_mod)
extract_features = features_mod.extract_features

DATA = ROOT / "data" / "places.csv"
df = pd.read_csv(DATA)
print("ROOT:", ROOT)
print("размер:", df.shape)

## Задание 1. Сбор датасета

**Как выбирал места:** Москва; из `moscow_places.csv` — заведения с высоким рейтингом (топ-18 кафе и топ-18 ресторанов после маппинга категорий). Культура — список известных музеев, парков, театров и достопримечательностей.

**Категории:** `cafe`, `restaurant`, `museum`, `park`, `theatre`, `attraction`.

**Источники:** Яндекс Карты / Бизнес (2022, учебный датасет); описания культуры — **Википедия (ru)** (`wiki_cache.json`).

**Лимиты:** до 18 объектов на категорию общепита; культура — фиксированный список в `build_dataset.py` (~43 объекта). Итого **50–100** мест.

**Поля:** название, категория, описание, отзывы (для еды — рейтинг и цены из Яндекс; текстовых отзывов в выгрузке нет), город, `source`.

In [ ]:
df.head(10)

In [ ]:
print(df["category"].value_counts())
print()
print(df["source"].value_counts())

## Задание 2. Семантические признаки

| Признак | Зачем |
|--------|--------|
| `experience` | еда / культура / прогулка / развлечения — тип опыта |
| `atmosphere` | уют, шум, романтика, работа |
| `visit_context` | семья, друзья, свидание, работа |
| `budget` | грубая ценовая категория |
| `time_of_day` | утро / день / вечер |
| `popularity` | прокси по длине текста и упоминанию рейтинга |

Нужны, чтобы отвечать на интент пользователя («уютное кафе для работы»), а не только на OSM-категорию.

## Задание 3. Извлечение признаков

- **Метод:** rule-based (словари русских ключевых слов), код в `src/features.py`.
- **Почему:** мало размеченных данных, нужна прозрачность и скорость.
- **Альтернативы:** LLM zero-shot; fine-tune BERT; только эмбеддинги без явных тегов.

In [ ]:
df_enriched = pd.concat([df, df.apply(extract_features, axis=1, result_type="expand")], axis=1)
df_enriched.to_csv(ROOT / "data" / "places_enriched.csv", index=False)
df_enriched.head()

## Задание 4. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_enriched["category"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Распределение категорий")
df_enriched["budget"].value_counts().plot(kind="pie", autopct="%1.0f%%", ax=axes[1])
axes[1].set_title("Бюджет")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

atm = df_enriched["atmosphere"].str.split(",").str[0].str.strip()
ctx = df_enriched["visit_context"].str.split(",").str[0].str.strip()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
atm.value_counts().plot(kind="barh", ax=axes[0])
axes[0].set_title("Атмосфера")
ctx.value_counts().plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Контекст посещения")
plt.tight_layout()
plt.show()

sns.histplot(df_enriched["popularity"], bins=10, kde=True)
plt.title("Popularity")
plt.show()

## Задание 5. Семантический поиск

TF-IDF по тексту + извлечённым признакам, cosine similarity.

In [ ]:
cols = ["name", "description", "reviews", "experience", "atmosphere", "visit_context", "budget"]
corpus = df_enriched[cols].fillna("").agg(" ".join, axis=1)
vec = TfidfVectorizer(ngram_range=(1, 2))
X = vec.fit_transform(corpus)

def search(query, top_k=5):
    sims = cosine_similarity(vec.transform([query]), X).flatten()
    idx = sims.argsort()[::-1][:top_k]
    out = df_enriched.iloc[idx][["name", "category", "atmosphere", "visit_context", "budget"]].copy()
    out["score"] = sims[idx]
    return out

search("уютное кафе для работы с ноутбуком")

In [ ]:
queries = [
    "уютное кафе для работы",
    "романтический ужин",
    "куда сходить с детьми",
    "прогулка в парке",
    "современное искусство",
]
rows = []
for q in queries:
    for rank, (_, r) in enumerate(search(q, top_k=3).iterrows(), 1):
        rows.append({
            "query": q,
            "rank": rank,
            "name": r["name"],
            "category": r["category"],
            "score": round(float(r["score"]), 3),
        })
search_table = pd.DataFrame(rows)
search_table.to_csv(ROOT / "data" / "search_comparison.csv", index=False)
search_table

### Оценка поиска

- Запросы про **кофе/работу** чаще попадают в `cafe` — помогают слова «работа», «кофе» в тексте и признаки.
- **С детьми** — музеи/парки с контекстом «семья».
- **Современное искусство** — музеи с «современн» в описании Википедии.
- Слабые места: нет полноценных текстовых отзывов; TF-IDF не ловит все синонимы.

## Выводы

1. Собрать 50–100 мест из открытых источников реально; узкое место — **текстовые отзывы**.
2. Явные семантические теги + TF-IDF — нормальный прототип для conversational-гида.
3. Rule-based признаки легко объяснить пользователю.
4. Ограничения: смещение к центру/известным местам; для продакшена — эмбеддинги и геофильтр.